In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [3]:
words = open('names.txt','r').read().splitlines()

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [194]:
def build_dataset(words):

    block_size = 3  

    X, Y = [], []

    for w in words:
        
        context = [0] * block_size

        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)

            

            context = context[1:] + [ix] 

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y

In [195]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])


In [9]:
torch.cat(torch.unbind(emb,1),1)

tensor([[ 0.6574,  1.7146,  0.6574,  1.7146,  0.6574,  1.7146],
        [ 0.6574,  1.7146,  0.6574,  1.7146, -0.7374, -0.6671],
        [ 0.6574,  1.7146, -0.7374, -0.6671,  0.3461,  1.0372],
        ...,
        [ 1.4013,  2.2656,  1.4013,  2.2656, -0.0800, -0.5731],
        [ 1.4013,  2.2656, -0.0800, -0.5731,  1.4013,  2.2656],
        [-0.0800, -0.5731,  1.4013,  2.2656,  1.2555, -1.4783]])

In [ ]:
g = torch.Generator().manual_seed(2147483647)  
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [276]:
sum(p.nelement() for p in parameters)  # number of parameters in total

11897

In [277]:
for p in parameters:
    p.requires_grad = True


In [314]:
for _ in range(10000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    emb = C[Xtr[ix]]  

    h = torch.tanh(emb.view(-1, W1.shape[0]) @ W1 + b1)  

    logits = h @ W2 + b2  

    loss = F.cross_entropy(logits, Ytr[ix])

    

    # backward pass
    for p in parameters:
        p.grad = None

    loss.backward()

    # update
    for p in parameters:
        p.data += -0.0000001 * p.grad
    
print(loss.item())



1.8561724424362183


In [316]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1,W1.shape[0])@W1 + b1)
logits = h@W2 + b2
counts = logits.exp()
prob = counts/counts.sum(1,keepdims = True)
loss = -prob[torch.arange(Xdev.shape[0]),Ydev].log().mean()
loss


tensor(2.1508, grad_fn=<NegBackward0>)

In [328]:
# Generate names

for _ in range(20):

    out = []
    context = [0] * block_size

    while True:

        # Forward pass
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2

        # Convert logits to probabilities
        probs = F.softmax(logits, dim=1)

        # Sample next character
        ix = torch.multinomial(probs, num_samples=1).item()

        # Stop if special end character is generated
        if ix == 0:
            break

        out.append(itos[ix])

        # Update context
        context = context[1:] + [ix]

    print(''.join(out))

ble
viy
anmelli
blanifbyavin
shengianquetterron
preson
jhett
claidaralyn
elmarceleigh
sormy
shi
delon
jadiel
kendray
keri
kyrdael
nah
keel
nachamliianne
melynn
